In [17]:
import pulp
import pandas as pd
from pulp import GUROBI

#Parameters
commodities = 4
vertices = 5
edges = {
    #edge: [Cost,Capacity]
    (1,2): [1,20],
    (1,3): [1,10],
    (2,3): [2,10],
    (2,4): [4,20],
    (3,4): [8,40],
    (3,5): [5,10],
    (4,5): [3,30],
}
supplies = {
    #(source node,commodity): quantity
    (1,0): 15,
    (1,1): 5,
    (2,2): 10,
    (3,3): 5
}
demands = {
    #(destination node,commodity): quantity
    (4,0): 15,
    (5,1): 5,
    (5,2): 10,
    (5,3): 5
}

mcf = pulp.LpProblem('Multi commodity flow',pulp.LpMinimize)

#Define Decision variables
#Flow qty variable
f = {(e,c): pulp.LpVariable('f_'+str(e)+str(c),cat = 'Continuous',lowBound = 0) \
     for e in edges for c in range(commodities)}


#Define objective function
mcf += pulp.lpSum(f[e,c]*edges[e][0] for e in edges for c in range(commodities))

#Capacity constraint
for e in edges:
    mcf += pulp.lpSum(f[e,c] for c in range(commodities)) <= edges[e][1]

#Flow constraints
#Outflow - Inflow == Supply - Demand
for c in range(commodities):
    for i in range(1,vertices+1):
        mcf += pulp.lpSum(f[(i,j),c] if (i,j) in edges else 0 \
                          for j in range(1,vertices+1)) \
        - pulp.lpSum(f[(j,i),c] if (j,i) in edges else 0 \
                                                                                             for j in range(1,vertices+1)) \
        == pulp.lpSum(supplies[i,c] if (i,c) in supplies else 0) \
        - pulp.lpSum(demands[i,c] if (i,c) in demands else 0)
#Solve
mcf.solve(GUROBI())



Gurobi Optimizer version 10.0.0 build v10.0.0rc2 (win64)

CPU model: Intel(R) Core(TM) i5-9300HF CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 27 rows, 28 columns and 84 nonzeros
Model fingerprint: 0x1f794423
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 8e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 4e+01]
Presolve removed 20 rows and 17 columns
Presolve time: 0.01s
Presolved: 7 rows, 11 columns, 22 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.8993600e+02   5.016000e+00   0.000000e+00      0s
       4    2.2000000e+02   0.000000e+00   0.000000e+00      0s

Solved in 4 iterations and 0.06 seconds (0.00 work units)
Optimal objective  2.200000000e+02
Gurobi status= 2


1

In [18]:
for flow in f:
    if f[flow].value()>0:
        print(flow, f[flow].value())

((1, 2), 0) 10.0
((1, 3), 0) 5.0
((1, 3), 1) 5.0
((2, 4), 0) 10.0
((2, 4), 2) 10.0
((3, 4), 0) 5.0
((3, 5), 1) 5.0
((3, 5), 3) 5.0
((4, 5), 2) 10.0
